# ChaCha20

ChaCha is a **Stream Cipher**. That means, instead of scrabling the message directly, it first generates a long stream of random-looking bytes and then mixes the message with that stream.

As per the RFC-8439:

```txt
   ChaCha20 successively calls the ChaCha20 block function, with the
   same key and nonce, and with successively increasing block counter
   parameters.  ChaCha20 then serializes the resulting state by writing
   the numbers in little-endian order, creating a keystream block.
   Concatenating the keystream blocks from the successive blocks forms a
   keystream.  The ChaCha20 function then performs an XOR of this
   keystream with the plaintext.
```

It can be thought of as:

```bash
    Message:     HELLO
    Keystream:   Qx9@#
    Result:      (garbage)
```

The inputs to ChaCha20 are:
1. A 256-bit Key
2. A 96-bit nonce
3. A 32-bit initial counter
4. An arbitrary-length plaintext

The output is an encrypted message or "ciphertext", of the same length.

$\fbox{Note:}$ **Why aren't we allowed to reuse a number used once?**  
    &emsp;This is beacause if we reuse the same key paired with nonce, we get the same keystream thereby allowing attackers to deduce information. Since the message is XOR with the keystream.

### The Key

The key is the only secret in this algorithm. It determines:
- Who can generate the same keystream
- Who can decrypt the message

#### Properties:

- 256 bits
- Must remain secret
- Can be reused across many messages safely

The key is like the identifier of the encryption.

### The Nonce

The nonce ensures that the same key never reproduces the same keystream twice.

#### Properties:

- 96 bits
- Must be unique per key
- Can be public
- Random or sequential

The nonce is similar to the message ID.

### The Counter

ChaCha20 generates keystream in 64-byte chunks. The counter ensures that each chunk gets a different keystream, even within the same message.

## The Algorithm

In [2]:
# Utility
from typing import List
import struct

def rotl32(value: int, shift: int) -> int:
    """Rotate a 32-bit integer left"""
    return ((value << shift) & 0xffffffff) | (value >> (32 - shift))

def print_state(state: List[int], title: str):
    print(f"\n--- {title} ---")
    for i in range(4):
        row = state[i*4:(i+1)*4]
        print(" ".join(f"{x:08x}" for x in row))

### 1. Build the starting table

ChaCha20 starts by building a table of 16 numbers in 4x4 grid.

```bin
[ fixed words     ]   ← always the same
[ key words       ]   ← your secret
[ key words       ]
[ counter + nonce ]
```

|C|C|C|C|
|---|---|---|---|
|**K**|**K**|**K**|**K**|
|**K**|**K**|**K**|**K**|
|**B**|**N**|**N**|**N**|

Where, each cell is a 32 bit word

```bash
C - Constant
K - Key
B - 32-bit Block Counter
N - 96-bit Nonce split across 3 words
```

In [3]:
def build_state(key: bytes, counter: int, nonce: bytes) -> List[int]:
    """
    Builds the initial 4x4 ChaCha20 state
    """

    constants = b"expand 32-byte k"

    state = []
    state += struct.unpack("<4I", constants)
    state += struct.unpack("<8I", key)
    state.append(counter)
    state += struct.unpack("<3I", nonce)

    return list(state)

### 2. Copy the table

ChaCha20 immediately makes a copy of this table where one copy stays unchanged and the other copy gets aggressively scrambled.

### 3. Scramble the table

The algorithm now repeatedly scrambles the numbers in the table 20 times i.e 10 double rounds. Each double round = 1 column round and 1 diagonal round. It acieves this by adding numbers, flipping bits (XOR) and rotating bits in a circle.

#### The Mixing Operation

ChaCha20 repeatedly takes four numbers at a time and mixes them so that:
- Changing one bit affects many others
- Patterns disappear
- Output looks random
This operation is called a quarter round. Each quarter round operates on a fixed group of four words (either a column or diagonal).

#### Rounds

ChaCha20 does:
- Mix columns
- Then mix diagonals
- Then repeat

This happens 20 times. Why 20?
- Fewer rounds are faster but riskier
- 20 gives a huge safety margin
- No known attack breaks full 20 rounds

After these rounds, the table looks completely unrelated to the original inputs.

In [4]:
def quarter_round(state: List[int], a: int, b: int, c: int, d: int):
    """
    ChaCha20 quarter round:
    Operates on four indices of the state
    """

    state[a] = (state[a] + state[b]) & 0xffffffff
    state[d] ^= state[a]
    state[d] = rotl32(state[d], 16)

    state[c] = (state[c] + state[d]) & 0xffffffff
    state[b] ^= state[c]
    state[b] = rotl32(state[b], 12)

    state[a] = (state[a] + state[b]) & 0xffffffff
    state[d] ^= state[a]
    state[d] = rotl32(state[d], 8)

    state[c] = (state[c] + state[d]) & 0xffffffff
    state[b] ^= state[c]
    state[b] = rotl32(state[b], 7)

def chacha20_block(key: bytes, counter: int, nonce: bytes) -> bytes:
    """
    Produces one 64-byte keystream block
    """

    state = build_state(key, counter, nonce)
    working_state = state.copy()

    print_state(state, "Initial State")

    for round_num in range(1, 11):
        # Column rounds
        quarter_round(working_state, 0, 4, 8, 12)
        quarter_round(working_state, 1, 5, 9, 13)
        quarter_round(working_state, 2, 6, 10, 14)
        quarter_round(working_state, 3, 7, 11, 15)

        # Diagonal rounds
        quarter_round(working_state, 0, 5, 10, 15)
        quarter_round(working_state, 1, 6, 11, 12)
        quarter_round(working_state, 2, 7, 8, 13)
        quarter_round(working_state, 3, 4, 9, 14)

        print_state(working_state, f"After Double Round {round_num}")

    # Feed-forward
    for i in range(16):
        working_state[i] = (working_state[i] + state[i]) & 0xffffffff

    print_state(working_state, "After Feed-Forward")

    # Serialize
    keystream = b"".join(struct.pack("<I", word) for word in working_state)

    print("\nKeystream (64 bytes):")
    print(keystream.hex())

    return keystream


### 4. Add the original table back in

ChaCha20 now adds the scrambled table to the original table number by number. This feed-forward step prevents the internal permutation from being reversible.


### 5. Turn the table into bytes

The final table is now converted into bytes and is then produced as 64 bytes of output. These 64 bytes are the pure keystream.


### 6. Encrypt Message

ChaCha20 now combines the keystream and plaintext using XOR byte-by-byte.
$$
\text{Encrypted Data} = \text{Message} \oplus \text{Keystream}
$$


In [5]:
def chacha20_xor(data: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    result = b""
    block_counter = counter

    for i in range(0, len(data), 64):
        block = data[i:i+64]
        keystream = chacha20_block(key, block_counter, nonce)

        result_block = bytes(b ^ k for b, k in zip(block, keystream))
        result += result_block
        block_counter += 1

    return result

def chacha20_encrypt(plaintext: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    print("\n=== ENCRYPTION ===")
    return chacha20_xor(plaintext, key, nonce, counter)

def chacha20_decrypt(ciphertext: bytes, key: bytes, nonce: bytes, counter: int = 1) -> bytes:
    print("\n=== DECRYPTION ===")
    return chacha20_xor(ciphertext, key, nonce, counter)


### 7. Move to Next Block

If the message is longer than 64 bytes:
- Increase the counter
- Build a new table
- Scramble again
- Produce the next keystream block
- Repeat

In [6]:
key = b"\x00" * 32
nonce = b"\x00" * 12
message = b"Hello ChaCha20! This is a fully traced encryption demo."

print("\nOriginal Message:")
print(message)


Original Message:
b'Hello ChaCha20! This is a fully traced encryption demo.'


In [7]:
encrypted = chacha20_encrypt(message, key, nonce)

print("\nFinal Ciphertext:")
print(encrypted.hex())


=== ENCRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
00000000 00000000 00000000 00000000
00000000 00000000 00000000 00000000
00000001 00000000 00000000 00000000

--- After Double Round 1 ---
d35fd249 1be820a9 d31f88da 8e7f397f
c1bf4b8a feb74a18 a1f753eb f91e808c
578cda2e 042c17b1 010a06a9 49f7792a
2bbe8409 c63023e6 ab8f9939 87d62152

--- After Double Round 2 ---
bc389fa8 b9ba7055 3085e7bd 009ee8cf
0ab5eecd 3111471a 86b484ee 7f517f6d
47323b48 1289b2d8 f81f7778 ad21d4ee
6fb0164a add42d2e 81ed68a2 66145871

--- After Double Round 3 ---
05ef5f24 0a23c120 f0ab6935 e071f61f
e2a62d59 57de135b 25e917b1 97526658
cf306ca2 5dc44a81 e020206c c89e2f10
5d348f46 50e06f03 48e9dd8c 7cb1007c

--- After Double Round 4 ---
ac8a366d 1ea1417c f378dd8d 42858c8d
1258fdc0 aaa2f959 8f0ff2dc 6ba266d5
38ec3250 98dac5bb 566f0cee 652a878b
25bf8a9f bb21eb1d d8e5564b aa681e82

--- After Double Round 5 ---
b42c1d68 2a9852a5 7d93a117 44f144cd
e6eb9a9e 15e413ae ea58f506 6af9e90c
6a11b951 550c05

In [8]:
decrypted = chacha20_decrypt(encrypted, key, nonce)

print("\nDecrypted Message:")
print(decrypted)


=== DECRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
00000000 00000000 00000000 00000000
00000000 00000000 00000000 00000000
00000001 00000000 00000000 00000000

--- After Double Round 1 ---
d35fd249 1be820a9 d31f88da 8e7f397f
c1bf4b8a feb74a18 a1f753eb f91e808c
578cda2e 042c17b1 010a06a9 49f7792a
2bbe8409 c63023e6 ab8f9939 87d62152

--- After Double Round 2 ---
bc389fa8 b9ba7055 3085e7bd 009ee8cf
0ab5eecd 3111471a 86b484ee 7f517f6d
47323b48 1289b2d8 f81f7778 ad21d4ee
6fb0164a add42d2e 81ed68a2 66145871

--- After Double Round 3 ---
05ef5f24 0a23c120 f0ab6935 e071f61f
e2a62d59 57de135b 25e917b1 97526658
cf306ca2 5dc44a81 e020206c c89e2f10
5d348f46 50e06f03 48e9dd8c 7cb1007c

--- After Double Round 4 ---
ac8a366d 1ea1417c f378dd8d 42858c8d
1258fdc0 aaa2f959 8f0ff2dc 6ba266d5
38ec3250 98dac5bb 566f0cee 652a878b
25bf8a9f bb21eb1d d8e5564b aa681e82

--- After Double Round 5 ---
b42c1d68 2a9852a5 7d93a117 44f144cd
e6eb9a9e 15e413ae ea58f506 6af9e90c
6a11b951 550c05

In [9]:
import secrets

key = secrets.token_bytes(32)
nonce = secrets.token_bytes(12)

def hexify(b: bytes) -> str:
    return b.hex()

plaintext_str = input("Enter plaintext: ")
plaintext = plaintext_str.encode("utf-8")

print("\nGenerated Key (hex):")
print(hexify(key))

print("\nGenerated Nonce (hex):")
print(hexify(nonce))


Generated Key (hex):
4635e89e1c825c1ca149a8da68194055d4beb8bdf321a6893ef6e63168474e69

Generated Nonce (hex):
338d08e9fc554e09504c2b90


In [10]:
ciphertext = chacha20_encrypt(
    plaintext=plaintext,
    key=key,
    nonce=nonce,
    counter=1
)

print("\nFinal Ciphertext (hex):")
print(hexify(ciphertext))


=== ENCRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
9ee83546 1c5c821c daa849a1 55401968
bdb8bed4 89a621f3 31e6f63e 694e4768
00000001 e9088d33 094e55fc 902b4c50

--- After Double Round 1 ---
2f9d8cab 89cb9a9e 3faf394b 2ac93471
6ee0fc1e 82b3fe89 12225a2a 903b4f11
9715631c b2fd8200 375c2ba8 e410075c
78165484 892f3272 a62b2b50 904584e1

--- After Double Round 2 ---
18d50716 521806c5 bf321d71 4c96016d
d97cf6b8 20876976 5bd0b517 0e61325c
d80c0d6f a56df473 1d257973 e078025e
8c1041f8 e30ae354 b8390f0a e59e06e0

--- After Double Round 3 ---
33c3ef7b aa5df78a b92f892c 4da57ecc
799bf2b2 521d81fb a037813e abe8523c
48569fd9 0f730145 52207608 efd963fe
b5f4b2a0 32f9c2d5 f18026a1 ec791e7d

--- After Double Round 4 ---
207b55c3 6c9d8d02 ec531851 b55fbe4b
c9e1ab54 0ef7d6d7 855744d7 2a785b30
d3daf89a 38a3d07e 3b3ec0cd 6d5e46ac
22a90227 2c2f196c 4335e4ae 8be13ca0

--- After Double Round 5 ---
a90ca714 4d2c39ee 84da1fc6 4ad67d27
616cea77 074c25cc ebc065a5 6d72a386
3f343ad0 8f1579

In [11]:
decrypted = chacha20_decrypt(
    ciphertext=ciphertext,
    key=key,
    nonce=nonce,
    counter=1
)

print("\nDecrypted Plaintext:")
print(decrypted.decode("utf-8"))


=== DECRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
9ee83546 1c5c821c daa849a1 55401968
bdb8bed4 89a621f3 31e6f63e 694e4768
00000001 e9088d33 094e55fc 902b4c50

--- After Double Round 1 ---
2f9d8cab 89cb9a9e 3faf394b 2ac93471
6ee0fc1e 82b3fe89 12225a2a 903b4f11
9715631c b2fd8200 375c2ba8 e410075c
78165484 892f3272 a62b2b50 904584e1

--- After Double Round 2 ---
18d50716 521806c5 bf321d71 4c96016d
d97cf6b8 20876976 5bd0b517 0e61325c
d80c0d6f a56df473 1d257973 e078025e
8c1041f8 e30ae354 b8390f0a e59e06e0

--- After Double Round 3 ---
33c3ef7b aa5df78a b92f892c 4da57ecc
799bf2b2 521d81fb a037813e abe8523c
48569fd9 0f730145 52207608 efd963fe
b5f4b2a0 32f9c2d5 f18026a1 ec791e7d

--- After Double Round 4 ---
207b55c3 6c9d8d02 ec531851 b55fbe4b
c9e1ab54 0ef7d6d7 855744d7 2a785b30
d3daf89a 38a3d07e 3b3ec0cd 6d5e46ac
22a90227 2c2f196c 4335e4ae 8be13ca0

--- After Double Round 5 ---
a90ca714 4d2c39ee 84da1fc6 4ad67d27
616cea77 074c25cc ebc065a5 6d72a386
3f343ad0 8f1579

In [18]:
# Encrypt the plaintext.txt and store as ciphertext_ChaCha20.txt
with open('../../plaintext.txt', 'rb') as f_in:
    plaintext_file = f_in.read()

# Ensure key is exactly 32 bytes (pad or truncate as needed)
_key = b"ThisIsA32ByteKeyForChaCha20"
key = (_key + b'\x00' * 32)[:32]

# Ensure nonce is exactly 12 bytes
nonce = b"12ByteNonce!"
if len(nonce) != 12:
    raise ValueError("Nonce must be 12 bytes long")

ciphertext_file = chacha20_encrypt(plaintext=plaintext_file, key=key, nonce=nonce)

# Write ciphertext to file
with open('ciphertext_ChaCha20.txt', 'wb') as f_out:
    f_out.write(ciphertext_file)
print("Wrote ciphertext to ciphertext_ChaCha20.txt")


=== ENCRYPTION ===

--- Initial State ---
61707865 3320646e 79622d32 6b206574
73696854 33417349 74794232 79654b65
43726f46 68436168 00303261 00000000
00000001 79423231 6f4e6574 2165636e

--- After Double Round 1 ---
cc5f1d57 f4dea57c 0e686814 7fe737e4
6dae1bee 24caf758 f8a3cbd8 ad548078
38bf0e0b c1e9d71d 39fb58d9 401111c9
d781c04f 71377673 da026501 a18635fa

--- After Double Round 2 ---
07e6df39 95f61bb5 30c6dca9 02aa23d5
7489edfa bb5e55b0 541a7502 4147bdd3
418c087f 591be588 0e446a45 3ab51036
6e524fe4 5f05aca2 c86a9551 7fca9fd8

--- After Double Round 3 ---
2ca945df 0a2de32a ccfe65c6 371c5d20
f73c662a e75ff937 0a60ecb7 b7709513
f0d68eb6 2a43d109 d03aa490 24544d34
b96d5d3f 12ec872b 520e7812 d9554713

--- After Double Round 4 ---
f2e73103 eca5050f 50a5254b ea582684
dcd05272 c51954ae 33dd5263 3ac8f3fd
e1ee2e39 9a5003a2 c2df7ce3 cf6d35e1
a606e24d 953913da 8f97aeb1 c4cc7cda

--- After Double Round 5 ---
b3c44d51 f6e02434 93a6815d 081dd977
44adbdf3 fb55db10 f57fd1ae 3a02e8f3
dffcc8a9 bdc669

## Entropy Analysis

This section measures Shannon entropy to quantify unpredictability of text. For a discrete message with symbol probabilities $p_i$:

$$
H = -\sum_i p_i \log_2 p_i \quad\text{(bits per symbol)}
$$

- *Plaintext entropy*: reflects language structure and redundancy (typically much less than maximum possible for the symbol set).
- *Ciphertext entropy*: good encryption increases entropy toward the maximum for the symbol alphabet used (making symbols appear uniform).
- Interpretation:
  - ciphertext_entropy >> plaintext_entropy $\rarr$ ciphertext hides structure well.
  - ciphertext_entropy ≈ plaintext_entropy $\rarr$ potential information leakage.

In [19]:
# Entropy Analysis
from collections import Counter
import math
def calculate_entropy(data: bytes) -> float:
    if not data:
        return 0.0

    byte_counts = Counter(data)
    total_bytes = len(data)

    entropy = 0.0
    for count in byte_counts.values():
        probability = count / total_bytes
        entropy -= probability * math.log2(probability)

    return entropy

entropy_plaintext = calculate_entropy(plaintext_file)
print(f"Entropy of plaintext: {entropy_plaintext:.4f} bits per byte")
entropy_value = calculate_entropy(ciphertext_file)
print(f"Entropy of ciphertext: {entropy_value:.4f} bits per byte")

Entropy of plaintext: 4.5461 bits per byte
Entropy of ciphertext: 7.9294 bits per byte
